In [ ]:
# -*- coding: utf-8 -*-
"""WESAD Model - ECG ONLY

Automatically generated by Colab.
"""

!pip install neurokit2 -q

import numpy as np
import pandas as pd
import pickle
import os
import joblib
import neurokit2 as nk
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from google.colab import drive
from google.colab import files

# Mount Drive
drive.mount('/content/drive')
base_path = '/content/drive/MyDrive/WESAD/'
subjects = ['S2','S3','S4','S5','S6','S7','S8']

all_features = []
all_labels = []

# ---------------------------------------------------------
# 1. Feature Extraction (ONLY ECG/HRV)
# ---------------------------------------------------------
def extract_features(ecg, labels, sampling_rate=700, window_size=10):
    features = []
    targets = []
    step = sampling_rate * window_size

    for i in range(0, len(ecg) - step, step):
        ecg_w = ecg[i:i+step]
        label_w = labels[i:i+step]

        # Majority vote for the window's label
        label = int(np.round(np.mean(label_w)))

        try:
            # Clean and extract peaks
            ecg_clean = nk.ecg_clean(ecg_w, sampling_rate=sampling_rate)
            peaks, _ = nk.ecg_peaks(ecg_clean, sampling_rate=sampling_rate)

            # Calculate Heart Rate
            rate = nk.ecg_rate(peaks, sampling_rate=sampling_rate)
            hr_mean = np.mean(rate) if len(rate) > 0 else 0
            hr_std = np.std(rate) if len(rate) > 0 else 0

            # Calculate HRV Time-Domain Features
            hrv = nk.hrv_time(peaks, sampling_rate=sampling_rate)
            rmssd = hrv["HRV_RMSSD"].values[0] if not hrv.empty else 0
            sdnn = hrv["HRV_SDNN"].values[0] if not hrv.empty else 0

        except:
            # Fallback if signal is too noisy to extract peaks
            hr_mean, hr_std, rmssd, sdnn = 0, 0, 0, 0

        # Append ONLY the 4 ECG features
        features.append([hr_mean, hr_std, rmssd, sdnn])
        targets.append(label)

    return features, targets

# ---------------------------------------------------------
# 2. Process WESAD Dataset
# ---------------------------------------------------------
for subject in subjects:
    print("Processing:", subject)
    file_path = os.path.join(base_path, subject, f"{subject}.pkl")

    if not os.path.exists(file_path):
        print("Missing:", file_path)
        continue

    with open(file_path, 'rb') as f:
        data = pickle.load(f, encoding='latin1')

    # Extract only ECG and Labels
    chest = data['signal']['chest']
    ecg = chest['ECG'].flatten()
    labels = data['label']

    # Keep only calm (1) & stress (2)
    idx = np.where((labels == 1) | (labels == 2))
    ecg = ecg[idx]
    labels = labels[idx]

    # Convert to Binary: 0 (Calm) and 1 (Stress)
    labels_binary = np.where(labels == 2, 1, 0)

    # Extract features using only ECG
    feats, labs = extract_features(ecg, labels_binary)

    all_features.extend(feats)
    all_labels.extend(labs)

print("\nTotal samples extracted:", len(all_features))

# ---------------------------------------------------------
# 3. Train Model
# ---------------------------------------------------------
X = np.array(all_features)
y = np.array(all_labels)

print("Class distribution:", dict(zip(*np.unique(y, return_counts=True))))

# Scale the 4 features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=42
)

# Train Random Forest
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight='balanced',
    random_state=42
)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# ---------------------------------------------------------
# 4. Save and Download
# ---------------------------------------------------------
joblib.dump(model, "stress_model_ecg_only.pkl")
joblib.dump(scaler, "scaler_ecg_only.pkl")
print("✅ New 4-Feature model saved!")

files.download("stress_model_ecg_only.pkl")
files.download("scaler_ecg_only.pkl")

# ---------------------------------------------------------
# 5. Sanity Check (Testing)
# ---------------------------------------------------------
print("\n🔍 Testing REAL CALM samples from dataset...\n")
count = 0
for i in range(len(X)):
    if y[i] == 0:
        sample_scaled = scaler.transform(X[i].reshape(1, -1))
        proba = model.predict_proba(sample_scaled)[0][1]
        pred = 1 if proba > 0.75 else 0
        print(f"Input [HR, HR_STD, RMSSD, SDNN]: {np.round(X[i], 2)}")
        print(f"Actual: 0, Predicted: {pred}, Prob (Stress): {proba:.2f}")
        print("-" * 30)
        count += 1
        if count == 3: break

print("\n🔍 Testing STRESS samples...\n")
count = 0
for i in range(len(X)):
    if y[i] == 1:
        sample_scaled = scaler.transform(X[i].reshape(1, -1))
        proba = model.predict_proba(sample_scaled)[0][1]
        pred = 1 if proba > 0.75 else 0
        print(f"Input [HR, HR_STD, RMSSD, SDNN]: {np.round(X[i], 2)}")
        print(f"Actual: 1, Predicted: {pred}, Prob (Stress): {proba:.2f}")
        print("-" * 30)
        count += 1
        if count == 3: break

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 688.9/688.9 kB 8.7 MB/s eta 0:00:00
Mounted at /content/drive
Processing: S2
Processing: S3
Processing: S4
Processing: S5
Processing: S6
Processing: S7
Processing: S8

Total samples extracted: 1262
Class distribution: {np.int64(0): np.int64(818), np.int64(1): np.int64(444)}

Accuracy: 0.849802371541502

Classification Report:
               precision    recall  f1-score   support

           0       0.88      0.88      0.88       164
           1       0.79      0.79      0.79        89

    accuracy                           0.85       253
   macro avg       0.84      0.84      0.84       253
weighted avg       0.85      0.85      0.85       253

✅ New 4-Feature model saved!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🔍 Testing REAL CALM samples from dataset...

Input [HR, HR_STD, RMSSD, SDNN]: [85.78  7.77 27.84 74.71]
Actual: 0, Predicted: 0, Prob (Stress): 0.55
------------------------------
Input [HR, HR_STD, RMSSD, SDNN]: [88.2   7.45 44.85 55.84]
Actual: 0, Predicted: 1, Prob (Stress): 0.94
------------------------------
Input [HR, HR_STD, RMSSD, SDNN]: [ 84.11  10.48  85.66 100.54]
Actual: 0, Predicted: 0, Prob (Stress): 0.52
------------------------------

🔍 Testing STRESS samples...

Input [HR, HR_STD, RMSSD, SDNN]: [ 83.31  14.93 281.25 219.84]
Actual: 1, Predicted: 1, Prob (Stress): 0.97
------------------------------
Input [HR, HR_STD, RMSSD, SDNN]: [82.77  4.05 38.47 38.71]
Actual: 1, Predicted: 1, Prob (Stress): 0.98
------------------------------
Input [HR, HR_STD, RMSSD, SDNN]: [84.61  6.02 45.52 54.89]
Actual: 1, Predicted: 1, Prob (Stress): 0.84
------------------------------
